# Dataset Preparation

This notebook validates and inspects the synthetic CSV dataset used by the intent classifier. It checks schema compatibility with `model_config.yaml`, summarizes label distributions, and creates train/validation/test split IDs.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

from intent_classifier.dataset import (
    RequestDataset,
    compute_label_distributions,
    split_ids,
    validate_dataset_frame,
)
from intent_classifier.settings import load_model_config, load_train_config

pd.set_option("display.max_columns", 80)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_CONFIG_PATH = PROJECT_ROOT / "intent_classifier/config/model_config.yaml"
TRAIN_CONFIG_PATH = PROJECT_ROOT / "intent_classifier/config/train_config.yaml"

model_config = load_model_config(MODEL_CONFIG_PATH)
train_config = load_train_config(TRAIN_CONFIG_PATH)
dataset_path = PROJECT_ROOT / train_config.dataset_csv

model_config.head_names, dataset_path

In [ ]:
df = pd.read_csv(dataset_path)
df.head()

In [ ]:
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print("Configured label columns:", len(model_config.label_columns))

missing_columns = sorted(set(model_config.label_columns) - set(df.columns))
extra_label_like_columns = sorted(
    column for column in df.columns
    if "__" in column and column not in model_config.label_columns
)

print("Missing configured label columns:", missing_columns)
print("Extra label-like columns:", extra_label_like_columns)

In [ ]:
validated_df = validate_dataset_frame(df, model_config)
dataset = RequestDataset(validated_df, model_config)
print("Dataset is valid.")
dataset[0]

In [ ]:
distributions = compute_label_distributions(validated_df, model_config)
distributions

In [ ]:
origin_domain = pd.crosstab(validated_df["origin"], validated_df["domain"])
origin_domain

In [ ]:
label_counts = []
for head in model_config.heads:
    for label in head.labels:
        column = f"{head.name}__{label}"
        label_counts.append(
            {
                "head": head.name,
                "label": label,
                "positives": int(validated_df[column].sum()),
                "prevalence": float(validated_df[column].mean()),
            }
        )

label_counts_df = pd.DataFrame(label_counts).sort_values(["head", "positives"], ascending=[True, False])
label_counts_df

In [ ]:
ax = label_counts_df.plot.barh(
    x="label",
    y="positives",
    figsize=(9, 6),
    title="Positive examples per label",
)
ax.invert_yaxis()

In [ ]:
splits = split_ids(validated_df, train_config.splits, seed=train_config.seed)
split_sizes = {
    "train": len(splits.train),
    "validation": len(splits.validation),
    "test": len(splits.test),
}
split_sizes

In [ ]:
def split_frame(indices):
    return validated_df.iloc[indices]

split_label_summary = []
for split_name, indices in {
    "train": splits.train,
    "validation": splits.validation,
    "test": splits.test,
}.items():
    split_df = split_frame(indices)
    for head in model_config.heads:
        active_examples = (split_df[list(head.label_columns)].sum(axis=1) > 0).sum()
        split_label_summary.append(
            {
                "split": split_name,
                "head": head.name,
                "rows": len(split_df),
                "rows_with_any_label": int(active_examples),
            }
        )

pd.DataFrame(split_label_summary)

In [ ]:
# Optional prepared output. The current synthetic dataset already validates cleanly.
WRITE_PREPARED_CSV = False
prepared_path = PROJECT_ROOT / "dataset/train_prepared.csv"

if WRITE_PREPARED_CSV:
    validated_df.to_csv(prepared_path, index=False)
    print(f"Wrote {prepared_path}")
else:
    print("Prepared CSV write skipped.")